In [ ]:
%pip install datasets sentence-transformers numpy scipy matplotlib pandas

In [ ]:
%pip install --upgrade httpcore httpx h11

In [ ]:
%pip install --upgrade brotli

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "sentence-transformers/wikipedia-en-sentences",
    split="train",
    streaming=True
)

In [3]:
for row in dataset.take(5):
    print(row["sentence"])

The film stars M. G. Ramachandran, Latha, Anjali Devi and V. S. Raghavan in lead roles.
Naarda plenirena is a species of moth in the family Noctuidae.
Sponsored by the American Federation of Labor and Congress of Industrial Organizations (AFL-CIO), labor groups paid the entire $40,000 cost of the memorial.
Since that election the Belfast Corporation Act 1896 had increased the borough from 6,800 to 16,500 acres, and the electorate from 39,603 to 47,294.
It was also included on their Best of Volume 1.


In [ ]:
from tqdm.auto import tqdm

N = 50_000
N_QUERIES = 500
SEED = 42

shuffled_dataset = dataset.shuffle(
    seed=SEED,
    buffer_size=100_000
)

sentences = []
seen = set()

with tqdm(total=N + N_QUERIES) as progress:
    for row in shuffled_dataset:
        text = row["sentence"].strip()

        if len(text.split()) >= 5 and text not in seen:
            sentences.append(text)
            seen.add(text)
            progress.update(1)

        if len(sentences) == N + N_QUERIES:
            break

print("Total sentences:", len(sentences))

In [5]:
database_sentences = sentences[:N]
query_sentences = sentences[N:]

print("Database:", len(database_sentences))
print("Queries:", len(query_sentences))

Database: 50000
Queries: 500


In [6]:
import json
from pathlib import Path

DATA_DIR = Path("wikipedia_nn_data")
DATA_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / "sentences.json", "w") as f:
    json.dump(database_sentences, f)

with open(DATA_DIR / "query_sentences.json", "w") as f:
    json.dump(query_sentences, f)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [8]:
sentence = database_sentences[0]

embedding = model.encode(
    sentence,
    normalize_embeddings=True
)

print("Sentence:", sentence)
print("Embedding shape:", embedding.shape)
print("First 10 coordinates:", embedding[:10])

Sentence: Gray Hill Esq and his wife, also traveling in 1895, met the Hameideh south of Madeba.
Embedding shape: (384,)
First 10 coordinates: [-0.01246414  0.10611856 -0.04010179  0.03397487 -0.08948258  0.02738277
 -0.07015882  0.05279549 -0.04784937 -0.02131057]


In [ ]:
X = model.encode(
    database_sentences,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Database shape:", X.shape)

In [ ]:
Q = model.encode(
    query_sentences,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Query shape:", Q.shape)

In [12]:
print("Database dtype:", X.dtype)

print("Database normalized:",
      np.allclose(np.linalg.norm(X, axis=1), 1))

print("Queries normalized:",
      np.allclose(np.linalg.norm(Q, axis=1), 1))

Database dtype: float32
Database normalized: True
Queries normalized: True


In [13]:
np.save(DATA_DIR / "embeddings.npy", X)
np.save(DATA_DIR / "queries.npy", Q)

print("Embeddings saved successfully.")

Embeddings saved successfully.
